# Stock Performance Analysis Tool
## ACC102 Mini Assignment - Track 4: Interactive Data Analysis Tool

**Author:** Student  
**Date:** April 2026  
**Data Source:** Yahoo Finance (via yfinance library)  
**Access Date:** April 2026

---

### 1. Problem Definition and Target User

**Analytical Problem:**  
Individual investors often struggle to compare stock performance across different companies and sectors. This tool aims to provide a comprehensive analysis of stock performance metrics, helping users make informed investment decisions.

**Target User:**  
- Individual investors seeking quick stock performance comparison
- Finance students learning about stock analysis
- Small business owners considering stock investments

**Business Relevance:**  
Stock market analysis is crucial for investment decision-making. Understanding price trends, volatility, and comparative performance helps investors optimize their portfolios and manage risk effectively.

### 2. Data Loading and Preparation

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Configure plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Libraries imported successfully!")

#### 2.1 Define Stock Universe

We will analyze a selection of major US stocks from different sectors to provide diversified insights:
- **Technology:** Apple (AAPL), Microsoft (MSFT)
- **Finance:** JPMorgan Chase (JPM), Bank of America (BAC)
- **Consumer:** Coca-Cola (KO), Walmart (WMT)
- **Healthcare:** Johnson & Johnson (JNJ)

In [ ]:
# Define stock tickers and company names
STOCKS = {
    'AAPL': 'Apple Inc.',
    'MSFT': 'Microsoft Corporation',
    'JPM': 'JPMorgan Chase & Co.',
    'BAC': 'Bank of America Corporation',
    'KO': 'The Coca-Cola Company',
    'WMT': 'Walmart Inc.',
    'JNJ': 'Johnson & Johnson'
}

# Define time period (1 year of historical data)
END_DATE = datetime.now()
START_DATE = END_DATE - timedelta(days=365)

print(f"Analysis Period: {START_DATE.strftime('%Y-%m-%d')} to {END_DATE.strftime('%Y-%m-%d')}")
print(f"Number of stocks: {len(STOCKS)}")
print(f"Stocks: {list(STOCKS.keys())}")

#### 2.2 Data Acquisition from Yahoo Finance

In [ ]:
# Download stock data
def download_stock_data(tickers, start_date, end_date):
    """
    Download historical stock data from Yahoo Finance.
    
    Parameters:
    -----------
    tickers : list
        List of stock ticker symbols
    start_date : datetime
        Start date for historical data
    end_date : datetime
        End date for historical data
    
    Returns:
    --------
    DataFrame : Multi-index DataFrame with stock data
    """
    print("Downloading stock data from Yahoo Finance...")
    
    # Download data for all tickers
    data = yf.download(
        tickers=list(tickers.keys()),
        start=start_date.strftime('%Y-%m-%d'),
        end=end_date.strftime('%Y-%m-%d'),
        progress=True
    )
    
    return data

# Download data
raw_data = download_stock_data(STOCKS, START_DATE, END_DATE)
print(f"\nData shape: {raw_data.shape}")
print(f"Data columns: {raw_data.columns.levels[0].tolist()}")

#### 2.3 Data Cleaning and Transformation

In [ ]:
# Extract adjusted close prices
def extract_close_prices(data, tickers):
    """
    Extract adjusted close prices and clean the data.
    
    Parameters:
    -----------
    data : DataFrame
        Raw stock data from Yahoo Finance
    tickers : dict
        Dictionary of ticker symbols and company names
    
    Returns:
    --------
    DataFrame : Cleaned close prices
    """
    # Get Adjusted Close prices
    if 'Adj Close' in data.columns.levels[0]:
        close_prices = data['Adj Close'].copy()
    else:
        close_prices = data['Close'].copy()
    
    # Check for missing values
    print("Missing values per stock:")
    missing = close_prices.isnull().sum()
    print(missing)
    
    # Forward fill missing values (if any)
    close_prices = close_prices.fillna(method='ffill')
    
    # Backward fill any remaining missing values
    close_prices = close_prices.fillna(method='bfill')
    
    print(f"\nData cleaned successfully!")
    print(f"Date range: {close_prices.index.min().strftime('%Y-%m-%d')} to {close_prices.index.max().strftime('%Y-%m-%d')}")
    print(f"Trading days: {len(close_prices)}")
    
    return close_prices

# Extract and clean close prices
close_prices = extract_close_prices(raw_data, STOCKS)
close_prices.head()

In [ ]:
# Save cleaned data to CSV for reproducibility
close_prices.to_csv('data/stock_prices.csv')
print("Data saved to data/stock_prices.csv")

### 3. Data Analysis

#### 3.1 Descriptive Statistics

In [ ]:
# Calculate descriptive statistics
def calculate_statistics(prices, tickers):
    """
    Calculate key statistical metrics for each stock.
    
    Parameters:
    -----------
    prices : DataFrame
        Stock close prices
    tickers : dict
        Dictionary of ticker symbols and company names
    
    Returns:
    --------
    DataFrame : Statistical summary
    """
    stats = pd.DataFrame()
    
    for ticker in tickers.keys():
        if ticker in prices.columns:
            series = prices[ticker]
            stats[ticker] = {
                'Company': tickers[ticker],
                'Start Price': series.iloc[0],
                'End Price': series.iloc[-1],
                'Mean Price': series.mean(),
                'Std Dev': series.std(),
                'Min Price': series.min(),
                'Max Price': series.max(),
                'Price Range': series.max() - series.min(),
                'Total Return (%)': ((series.iloc[-1] / series.iloc[0]) - 1) * 100
            }
    
    return stats.T

# Calculate statistics
stats_df = calculate_statistics(close_prices, STOCKS)
stats_df = stats_df.round(2)
stats_df

#### 3.2 Daily Returns Analysis

In [ ]:
# Calculate daily returns
def calculate_returns(prices):
    """
    Calculate daily percentage returns.
    
    Parameters:
    -----------
    prices : DataFrame
        Stock close prices
    
    Returns:
    --------
    DataFrame : Daily returns
    """
    returns = prices.pct_change() * 100  # Convert to percentage
    returns = returns.dropna()
    return returns

# Calculate returns
daily_returns = calculate_returns(close_prices)
print("Daily Returns Summary:")
daily_returns.describe().round(4)

In [ ]:
# Calculate risk metrics
def calculate_risk_metrics(returns, tickers):
    """
    Calculate risk metrics including volatility and Sharpe ratio.
    
    Parameters:
    -----------
    returns : DataFrame
        Daily returns
    tickers : dict
        Dictionary of ticker symbols and company names
    
    Returns:
    --------
    DataFrame : Risk metrics
    """
    risk_metrics = pd.DataFrame()
    
    # Risk-free rate assumption (annualized 5%)
    RISK_FREE_RATE = 0.05 / 252  # Daily risk-free rate
    
    for ticker in tickers.keys():
        if ticker in returns.columns:
            series = returns[ticker]
            
            # Annualized volatility
            annual_vol = series.std() * np.sqrt(252)
            
            # Annualized return
            annual_return = series.mean() * 252
            
            # Sharpe Ratio (simplified)
            sharpe = (annual_return - (RISK_FREE_RATE * 252 * 100)) / annual_vol
            
            # Maximum drawdown
            cumulative = (1 + series / 100).cumprod()
            running_max = cumulative.cummax()
            drawdown = (cumulative - running_max) / running_max * 100
            max_drawdown = drawdown.min()
            
            risk_metrics[ticker] = {
                'Company': tickers[ticker],
                'Daily Volatility (%)': series.std(),
                'Annual Volatility (%)': annual_vol,
                'Annual Return (%)': annual_return,
                'Sharpe Ratio': sharpe,
                'Max Drawdown (%)': max_drawdown
            }
    
    return risk_metrics.T

# Calculate risk metrics
risk_df = calculate_risk_metrics(daily_returns, STOCKS)
risk_df = risk_df.round(4)
risk_df

#### 3.3 Correlation Analysis

In [ ]:
# Calculate correlation matrix
correlation_matrix = daily_returns.corr()

# Display correlation matrix
print("Correlation Matrix of Daily Returns:")
correlation_matrix.round(4)

### 4. Data Visualization

#### 4.1 Stock Price Trends

In [ ]:
# Plot stock price trends
def plot_price_trends(prices, tickers, save_path=None):
    """
    Plot stock price trends over time.
    
    Parameters:
    -----------
    prices : DataFrame
        Stock close prices
    tickers : dict
        Dictionary of ticker symbols and company names
    save_path : str, optional
        Path to save the figure
    """
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Plot 1: All stocks normalized to 100 at start
    ax1 = axes[0, 0]
    normalized = (prices / prices.iloc[0]) * 100
    for ticker in tickers.keys():
        if ticker in prices.columns:
            ax1.plot(normalized.index, normalized[ticker], label=ticker, linewidth=2)
    ax1.set_title('Normalized Stock Performance (Base = 100)', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Normalized Price')
    ax1.legend(loc='upper left', fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Price distribution (box plot)
    ax2 = axes[0, 1]
    prices_melted = prices.melt(var_name='Ticker', value_name='Price')
    sns.boxplot(data=prices_melted, x='Ticker', y='Price', ax=ax2, palette='Set2')
    ax2.set_title('Stock Price Distribution', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Ticker')
    ax2.set_ylabel('Price ($)')
    ax2.tick_params(axis='x', rotation=45)
    
    # Plot 3: Total returns bar chart
    ax3 = axes[1, 0]
    total_returns = ((prices.iloc[-1] / prices.iloc[0]) - 1) * 100
    colors = ['green' if x > 0 else 'red' for x in total_returns]
    bars = ax3.bar(total_returns.index, total_returns, color=colors, alpha=0.7, edgecolor='black')
    ax3.set_title('Total Return (%) - 1 Year Period', fontsize=14, fontweight='bold')
    ax3.set_xlabel('Ticker')
    ax3.set_ylabel('Return (%)')
    ax3.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax3.tick_params(axis='x', rotation=45)
    
    # Add value labels on bars
    for bar, value in zip(bars, total_returns):
        height = bar.get_height()
        ax3.annotate(f'{value:.1f}%',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3),
                    textcoords="offset points",
                    ha='center', va='bottom', fontsize=10)
    
    # Plot 4: Volatility comparison
    ax4 = axes[1, 1]
    volatility = daily_returns.std() * np.sqrt(252)
    bars = ax4.bar(volatility.index, volatility, color='steelblue', alpha=0.7, edgecolor='black')
    ax4.set_title('Annualized Volatility (%)', fontsize=14, fontweight='bold')
    ax4.set_xlabel('Ticker')
    ax4.set_ylabel('Volatility (%)')
    ax4.tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    plt.show()

# Generate price trend plots
plot_price_trends(close_prices, STOCKS, 'figures/price_trends.png')

#### 4.2 Correlation Heatmap

In [ ]:
# Plot correlation heatmap
def plot_correlation_heatmap(corr_matrix, save_path=None):
    """
    Plot correlation heatmap of stock returns.
    
    Parameters:
    -----------
    corr_matrix : DataFrame
        Correlation matrix
    save_path : str, optional
        Path to save the figure
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
    sns.heatmap(corr_matrix, 
                mask=mask,
                annot=True, 
                fmt='.3f',
                cmap='RdYlGn',
                center=0,
                square=True,
                linewidths=0.5,
                cbar_kws={'shrink': 0.8},
                ax=ax)
    
    ax.set_title('Stock Returns Correlation Matrix', fontsize=14, fontweight='bold', pad=20)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    plt.show()

# Generate correlation heatmap
plot_correlation_heatmap(correlation_matrix, 'figures/correlation_heatmap.png')

#### 4.3 Risk-Return Analysis

In [ ]:
# Plot risk-return scatter
def plot_risk_return(returns, prices, tickers, save_path=None):
    """
    Plot risk-return scatter plot.
    
    Parameters:
    -----------
    returns : DataFrame
        Daily returns
    prices : DataFrame
        Stock prices
    tickers : dict
        Dictionary of ticker symbols and company names
    save_path : str, optional
        Path to save the figure
    """
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # Calculate annual return and volatility
    annual_return = returns.mean() * 252
    annual_vol = returns.std() * np.sqrt(252)
    
    # Create scatter plot
    colors = plt.cm.Set2(np.linspace(0, 1, len(tickers)))
    
    for i, ticker in enumerate(tickers.keys()):
        if ticker in returns.columns:
            ax.scatter(annual_vol[ticker], annual_return[ticker], 
                      s=200, c=[colors[i]], label=ticker, edgecolors='black', linewidth=1.5)
            ax.annotate(ticker, 
                       (annual_vol[ticker], annual_return[ticker]),
                       xytext=(10, 5), textcoords='offset points',
                       fontsize=11, fontweight='bold')
    
    ax.set_xlabel('Annualized Volatility (%)', fontsize=12)
    ax.set_ylabel('Annualized Return (%)', fontsize=12)
    ax.set_title('Risk-Return Profile', fontsize=14, fontweight='bold')
    ax.axhline(y=0, color='gray', linestyle='--', linewidth=0.8)
    ax.legend(loc='upper left', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    plt.show()

# Generate risk-return plot
plot_risk_return(daily_returns, close_prices, STOCKS, 'figures/risk_return.png')

#### 4.4 Moving Average Analysis

In [ ]:
# Calculate and plot moving averages
def plot_moving_averages(prices, ticker, windows=[20, 50, 200], save_path=None):
    """
    Plot stock price with moving averages.
    
    Parameters:
    -----------
    prices : DataFrame
        Stock close prices
    ticker : str
        Stock ticker symbol
    windows : list
        List of moving average windows
    save_path : str, optional
        Path to save the figure
    """
    fig, ax = plt.subplots(figsize=(14, 7))
    
    # Plot price
    ax.plot(prices.index, prices[ticker], label='Close Price', linewidth=2, color='black')
    
    # Plot moving averages
    colors = ['blue', 'orange', 'green']
    for window, color in zip(windows, colors):
        ma = prices[ticker].rolling(window=window).mean()
        ax.plot(prices.index, ma, label=f'{window}-day MA', linewidth=1.5, color=color, alpha=0.8)
    
    ax.set_title(f'{ticker} - Price and Moving Averages', fontsize=14, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Price ($)')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Figure saved to {save_path}")
    
    plt.show()

# Generate moving average plot for Apple
plot_moving_averages(close_prices, 'AAPL', save_path='figures/moving_averages.png')

### 5. Key Findings and Insights

Based on the analysis, we can derive the following insights:

In [ ]:
# Generate summary report
def generate_summary_report(stats_df, risk_df, correlation_matrix):
    """
    Generate a summary report of key findings.
    
    Parameters:
    -----------
    stats_df : DataFrame
        Statistical summary
    risk_df : DataFrame
        Risk metrics
    correlation_matrix : DataFrame
        Correlation matrix
    
    Returns:
    --------
    str : Summary report
    """
    report = []
    report.append("=" * 60)
    report.append("STOCK PERFORMANCE ANALYSIS - KEY FINDINGS")
    report.append("=" * 60)
    
    # Best and worst performers
    best_return = stats_df['Total Return (%)'].idxmax()
    worst_return = stats_df['Total Return (%)'].idxmin()
    
    report.append(f"\n1. PERFORMANCE HIGHLIGHTS:")
    report.append(f"   - Best Performer: {best_return} ({stats_df.loc[best_return, 'Total Return (%)']:.2f}%)")
    report.append(f"   - Worst Performer: {worst_return} ({stats_df.loc[worst_return, 'Total Return (%)']:.2f}%)")
    
    # Risk analysis
    safest = risk_df['Annual Volatility (%)'].idxmin()
    riskiest = risk_df['Annual Volatility (%)'].idxmax()
    
    report.append(f"\n2. RISK ANALYSIS:")
    report.append(f"   - Lowest Volatility: {safest} ({risk_df.loc[safest, 'Annual Volatility (%)']:.2f}%)")
    report.append(f"   - Highest Volatility: {riskiest} ({risk_df.loc[riskiest, 'Annual Volatility (%)']:.2f}%)")
    
    # Best Sharpe ratio
    best_sharpe = risk_df['Sharpe Ratio'].idxmax()
    report.append(f"   - Best Risk-Adjusted Return: {best_sharpe} (Sharpe: {risk_df.loc[best_sharpe, 'Sharpe Ratio']:.4f})")
    
    # Correlation insights
    # Find highest correlation pair
    corr_values = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
    max_corr = corr_values.max().max()
    max_corr_pair = corr_values.stack().idxmax()
    
    report.append(f"\n3. CORRELATION INSIGHTS:")
    report.append(f"   - Highest Correlation: {max_corr_pair[0]} & {max_corr_pair[1]} ({max_corr:.4f})")
    report.append(f"   - This suggests these stocks tend to move together")
    
    report.append("\n" + "=" * 60)
    
    return "\n".join(report)

# Generate and display summary
summary = generate_summary_report(stats_df, risk_df, correlation_matrix)
print(summary)

### 6. Save Analysis Results

In [ ]:
# Save all analysis results to CSV files
stats_df.to_csv('data/statistics_summary.csv')
risk_df.to_csv('data/risk_metrics.csv')
correlation_matrix.to_csv('data/correlation_matrix.csv')

print("All analysis results saved successfully!")
print("\nFiles created:")
print("- data/stock_prices.csv")
print("- data/statistics_summary.csv")
print("- data/risk_metrics.csv")
print("- data/correlation_matrix.csv")
print("- figures/price_trends.png")
print("- figures/correlation_heatmap.png")
print("- figures/risk_return.png")
print("- figures/moving_averages.png")

### 7. Conclusion

This notebook demonstrates a comprehensive stock performance analysis workflow:

1. **Data Acquisition**: Successfully downloaded historical stock data from Yahoo Finance
2. **Data Cleaning**: Handled missing values and prepared clean dataset
3. **Statistical Analysis**: Calculated key metrics including returns, volatility, and Sharpe ratios
4. **Visualization**: Created informative charts showing price trends, correlations, and risk-return profiles
5. **Insights**: Identified best/worst performers and correlation patterns

**For interactive exploration, please run the Streamlit app (`app.py`).**